# Unemployment Analysis in India: Monthly Trends and COVID-19 Impact
**Reproducible data science project | Pandas · NumPy · Matplotlib · Seaborn**

**Source:** [Kaggle — Unemployment in India](https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india). The source data must be downloaded before analysis; this notebook does not invent results. All displayed conclusions are calculated when cells execute. **Interpretation:** unweighted averages of state/area observations are not official national rates. The pandemic comparison is descriptive, not causal.

## 1. Setup and reproducibility

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5.5), "figure.dpi": 110, "axes.titlesize": 15, "axes.labelsize": 11})
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("visualizations"); FIG_DIR.mkdir(exist_ok=True)
SEED = 42

def show_plot(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()

def note(text):
    display(Markdown("**Observation:** " + str(text)))

def fmt(x, unit="%"):
    return "N/A" if pd.isna(x) else f"{x:.2f}{unit}"


## 2. Kaggle dataset download and loading
The download cell tries the public Kaggle endpoint first. If access is blocked, manually download and unzip the dataset into `data/` from the source URL, then rerun the notebook. No sample/synthetic data is substituted.

In [ ]:
import zipfile
import urllib.request

source_url = "https://www.kaggle.com/datasets/gokulrajkmv/unemployment-in-india"
zip_path = DATA_DIR / "kaggle_unemployment.zip"
existing = list(DATA_DIR.rglob("*.csv"))
if not existing:
    try:
        urllib.request.urlretrieve("https://www.kaggle.com/api/v1/datasets/download/gokulrajkmv/unemployment-in-india", zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(DATA_DIR)
        print("Dataset downloaded from Kaggle.")
    except Exception as exc:
        print(f"Automatic download unavailable: {type(exc).__name__}: {exc}")
        print(f"Download manually from {source_url} and extract the CSV into {DATA_DIR.resolve()}")

csv_files = list(DATA_DIR.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("Genuine Kaggle CSV missing. Download it into data/ and rerun from this cell.")
print("Available CSV files:", [str(f) for f in csv_files])
# Prefer the 2019–2020 rural/urban dataset over the separate 2020-only file.
chosen = next((f for f in csv_files if f.stem.strip().lower() == "unemployment in india"), csv_files[0])
print("Selected:", chosen)
df = pd.read_csv(chosen)
display(df.head(10), df.tail(10))
print("Shape:", df.shape, "\nColumns:", list(df.columns))
df.info()
display(df.describe(include="all"), df.isnull().sum(), df.nunique())
print("Exact duplicate rows:", df.duplicated().sum())


## 3. Data cleaning, validation, and feature engineering
Missing numeric values are filled only within the same state and area using the median if sufficiently supported. Rows lacking a valid date, state, or unemployment rate after attempted treatment are excluded and counted. Exact duplicates are removed; distinct rural/urban observations are retained.

In [ ]:
raw = df.copy()
df.columns = [re.sub(r"\s+", " ", str(c).strip()) for c in df.columns]
rename = {}
for c in df.columns:
    k = re.sub(r"[^a-z]", "", c.lower())
    if k in ("region", "state", "states"): rename[c] = "State"
    elif k == "date": rename[c] = "Date"
    elif "unemploymentrate" in k: rename[c] = "Unemployment_Rate"
    elif "labourparticipationrate" in k or "laborparticipationrate" in k: rename[c] = "Labour_Participation_Rate"
    elif "estimatedemployed" in k or k == "employed": rename[c] = "Employed"
    elif k == "area": rename[c] = "Area"
df = df.rename(columns=rename)
required = {"State", "Date", "Unemployment_Rate"}
if not required.issubset(df): raise ValueError(f"Missing required columns: {required-set(df.columns)}; found {list(df.columns)}")
if "Area" not in df: df["Area"] = "Not reported"
df["State"] = df["State"].astype("string").str.strip()
df["Area"] = df["Area"].astype("string").str.strip().str.title().fillna("Not reported")
df["Date"] = pd.to_datetime(df["Date"].astype(str).str.strip(), dayfirst=True, errors="coerce")
for col in ["Unemployment_Rate", "Employed", "Labour_Participation_Rate"]:
    if col in df:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
        if col != "Employed": df.loc[~df[col].between(0, 100), col] = np.nan
        else: df.loc[df[col] < 0, col] = np.nan
print("Missing values before treatment:")
display(df.isna().sum().to_frame("Missing"))
plt.figure(figsize=(12, 4)); sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="mako")
plt.title("Missing values before treatment"); show_plot("01_missingness")
note("Dark/light bands identify missing entries; inspect missingness by state and time before treating gaps. Missing unemployment is never replaced with zero.")
before = len(df)
df = df.drop_duplicates().dropna(subset=["State", "Date"])
for col in ["Unemployment_Rate", "Employed", "Labour_Participation_Rate"]:
    if col in df:
        # Group-specific median: avoid cross-state imputation.
        med = df.groupby(["State", "Area"], dropna=False)[col].transform("median")
        df[col] = df[col].fillna(med)
df = df.dropna(subset=["Unemployment_Rate"]).copy()
df["Year"] = df.Date.dt.year
df["Month"] = df.Date.dt.month
df["Month_Name"] = df.Date.dt.strftime("%B")
df["Year_Month"] = df.Date.dt.to_period("M").dt.to_timestamp()
df["COVID_Period"] = np.where(df.Date < pd.Timestamp("2020-03-01"), "Pre-COVID", "COVID-affected")
df["Lockdown_Focus"] = df.Year_Month.between(pd.Timestamp("2020-04-01"), pd.Timestamp("2020-05-01"))
if "Labour_Participation_Rate" in df:
    df["Employment_Rate"] = df["Labour_Participation_Rate"] * (1 - df["Unemployment_Rate"] / 100)
df = df.sort_values(["Date", "State", "Area"]).reset_index(drop=True)
df.to_csv("India_Unemployment_Cleaned.csv", index=False)
print("Removed rows (duplicates/invalid):", before-len(df))
print("Rows:", len(df), "States:", df.State.nunique(), "Period:", df.Date.min().date(), "to", df.Date.max().date())
display(df.head(), df.isna().sum().to_frame("Remaining missing"))


## 4. Descriptive statistics and unemployment distribution

In [ ]:
display(df[[c for c in ["Unemployment_Rate", "Employed", "Labour_Participation_Rate", "Employment_Rate"] if c in df]].describe().T)
fig, ax = plt.subplots(); sns.histplot(df, x="Unemployment_Rate", kde=True, bins=28, ax=ax)
ax.axvline(df.Unemployment_Rate.mean(), color="darkred", linestyle="--", label="Mean")
ax.set(title="Distribution of reported unemployment rates", xlabel="Unemployment rate (%)", ylabel="Observation count"); ax.legend()
show_plot("02_distribution")
note(f"The mean observed unemployment rate is {fmt(df.Unemployment_Rate.mean())}, versus median {fmt(df.Unemployment_Rate.median())}; these describe state-area observations, not a population-weighted national rate.")
fig, ax = plt.subplots(); sns.boxplot(x=df.Unemployment_Rate, ax=ax)
ax.set(title="Unemployment rate: spread and possible outliers", xlabel="Unemployment rate (%)")
show_plot("03_boxplot")
q1,q3 = df.Unemployment_Rate.quantile([.25,.75]); out=((df.Unemployment_Rate<q1-1.5*(q3-q1))|(df.Unemployment_Rate>q3+1.5*(q3-q1))).sum()
note(f"The interquartile range is {fmt(q3-q1, ' percentage points')}; {out} observations fall outside Tukey's 1.5×IQR fences. Such values are not automatically errors.")


## 5. Monthly unemployment trends
Monthly observations are averaged across available state/area records; this is **not an official India-wide unemployment rate**.

In [ ]:
monthly = df.groupby("Year_Month", as_index=False)["Unemployment_Rate"].mean().sort_values("Year_Month")
fig, ax = plt.subplots(); ax.plot(monthly.Year_Month, monthly.Unemployment_Rate, marker="o", lw=2.2)
ax.axvspan(pd.Timestamp("2020-04-01"), pd.Timestamp("2020-06-01"), color="orange", alpha=.17, label="Apr–May 2020 lockdown focus")
peak = monthly.loc[monthly.Unemployment_Rate.idxmax()]
ax.annotate(f"Peak {peak.Unemployment_Rate:.1f}%", (peak.Year_Month, peak.Unemployment_Rate), xytext=(8, 15), textcoords="offset points", arrowprops={"arrowstyle":"->"})
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y")); ax.tick_params(axis="x", rotation=40)
ax.set(title="Monthly mean unemployment rate across reporting observations", xlabel="Month", ylabel="Mean unemployment rate (%)"); ax.legend()
show_plot("04_monthly_trend")
note(f"The highest observed monthly mean is {fmt(peak.Unemployment_Rate)} in {peak.Year_Month:%B %Y}. Changes around lockdown are associations, not causal estimates.")


## 6. Top ten states by average unemployment rate

In [ ]:
state_avg = df.groupby("State")["Unemployment_Rate"].mean().sort_values(ascending=False)
top10 = state_avg.head(10)
display(top10.rename("Mean unemployment rate (%)").to_frame())
fig, ax = plt.subplots(figsize=(11,6)); sns.barplot(x=top10.values, y=top10.index, ax=ax, color="#3676a7")
for i,v in enumerate(top10.values): ax.text(v+.15, i, f"{v:.1f}%", va="center")
ax.set_xlim(0,top10.max()*1.18); ax.set(title="Top 10 states: mean unemployment rate (available months)", xlabel="Mean unemployment rate (%)", ylabel="State")
show_plot("05_top10_states")
note(f"{top10.index[0]} has the highest mean in this sample ({fmt(top10.iloc[0])}). Comparisons may reflect different reporting coverage and are not weighted by state population.")


## 7. Three-state monthly time-series comparison

In [ ]:
preferred = ["Maharashtra", "Uttar Pradesh", "Tamil Nadu"]
selected = [s for s in preferred if s in df.State.unique()]
selected += [s for s in state_avg.index if s not in selected][:max(0,3-len(selected))]
print("Selected states:", selected)
three = df[df.State.isin(selected)].groupby(["Year_Month","State"],as_index=False)["Unemployment_Rate"].mean()
fig, ax = plt.subplots(); sns.lineplot(data=three, x="Year_Month", y="Unemployment_Rate", hue="State", marker="o", ax=ax)
ax.axvspan(pd.Timestamp("2020-04-01"),pd.Timestamp("2020-06-01"),color="orange",alpha=.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y")); ax.tick_params(axis="x",rotation=40)
ax.set(title="Monthly unemployment trends: three selected states", xlabel="Month",ylabel="Mean unemployment rate (%)")
show_plot("06_three_states")
state_peaks=three.loc[three.groupby("State").Unemployment_Rate.idxmax(),["State","Year_Month","Unemployment_Rate"]]
display(state_peaks)
note("Monthly peaks by state are shown in the table. Compare their timing with April–May 2020; the plot does not establish a causal COVID effect.")


## 8. Pre-COVID vs COVID-affected comparison
**Pre-COVID:** before March 2020. **COVID-affected:** March 2020 onward. March is a transition month, not a complete lockdown month. A separate April–May comparison is provided below.

In [ ]:
period_means=df.groupby("COVID_Period")["Unemployment_Rate"].mean().reindex(["Pre-COVID","COVID-affected"])
if period_means.isna().any(): raise ValueError("Both pre-COVID and COVID-affected observations are needed for this comparison.")
change=period_means.iloc[1]-period_means.iloc[0]
relative=100*change/period_means.iloc[0] if period_means.iloc[0] else np.nan
display(period_means.rename("Mean unemployment (%)").to_frame())
fig,ax=plt.subplots(figsize=(8,5)); bars=ax.bar(period_means.index,period_means.values,color=["#287c82","#d27b36"])
ax.bar_label(bars,fmt="%.2f%%",padding=3)
ax.set_ylim(0,max(period_means)*1.2);ax.set(title="Pre-COVID vs COVID-affected: mean observed unemployment",ylabel="Mean unemployment rate (%)")
show_plot("07_period_comparison")
note(f"Mean unemployment changed from {fmt(period_means.iloc[0])} to {fmt(period_means.iloc[1])}, a change of {change:+.2f} percentage points ({relative:+.1f}% relative). This is a descriptive comparison across unequal periods.")
lockdown_mean=df.loc[df.Lockdown_Focus,"Unemployment_Rate"].mean()
print("April–May 2020 lockdown-focused mean:",fmt(lockdown_mean))


## 9. State-wise COVID-period changes and top ten increases

In [ ]:
state_period=df.pivot_table(index="State",columns="COVID_Period",values="Unemployment_Rate",aggfunc="mean")
state_period=state_period.dropna(subset=["Pre-COVID","COVID-affected"])
state_period["Change_pp"]=state_period["COVID-affected"]-state_period["Pre-COVID"]
changes=state_period.sort_values("Change_pp",ascending=False).head(10)
display(changes.round(2))
fig,ax=plt.subplots(figsize=(11,6));sns.barplot(x=changes.Change_pp,y=changes.index,color="#bc6b45",ax=ax)
ax.axvline(0,color="black",lw=.8);ax.set(title="Top 10 states: increase in mean unemployment during COVID-affected period",xlabel="Change (percentage points)",ylabel="State")
show_plot("08_top10_changes")
if len(changes): note(f"The largest observed increase is {changes.Change_pp.iloc[0]:+.2f} percentage points in {changes.index[0]}; differences in monthly coverage may affect this ranking.")
else: note("No states have data in both periods, so a state-by-state comparison is unavailable.")
fig,ax=plt.subplots(figsize=(10,5));state_period.loc[[s for s in selected if s in state_period.index],["Pre-COVID","COVID-affected"]].plot(kind="bar",ax=ax)
ax.set(title="Selected states: period comparison",ylabel="Mean unemployment rate (%)",xlabel="State");ax.tick_params(axis="x",rotation=0)
show_plot("09_selected_state_periods")
note("This grouped comparison highlights differences in the magnitude of period changes across the three selected states.")


## 10. Correlation heatmap: unemployment and employment indicators
`Employed` is a count, not an employment rate. If labour-force participation is available as a percentage, the implied employment-to-population rate is `LFPR × (1 − unemployment_rate/100)`. Correlations with this derived indicator are partly mechanical.

In [ ]:
cols=[c for c in ["Unemployment_Rate","Employed","Labour_Participation_Rate","Employment_Rate"] if c in df]
corr=df[cols].corr()
display(corr)
fig,ax=plt.subplots(figsize=(9,7));sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm",center=0,vmin=-1,vmax=1,linewidths=.5,ax=ax)
ax.set_title("Correlation of reported labour-market indicators")
show_plot("10_correlation")
if "Employment_Rate" in corr: note(f"The correlation of unemployment and derived employment rate is {corr.loc['Unemployment_Rate','Employment_Rate']:.2f}. This is not independent evidence because the derived rate mathematically includes unemployment.")
else: note("Employment rate cannot be derived without labour-force participation; estimated employed counts should not be mislabelled as a rate.")


## 11. State × month unemployment heatmap

In [ ]:
matrix=df.pivot_table(index="State",columns="Year_Month",values="Unemployment_Rate",aggfunc="mean")
fig,ax=plt.subplots(figsize=(max(11,matrix.shape[1]*.8),max(7,matrix.shape[0]*.29)))
sns.heatmap(matrix,cmap="YlOrRd",linewidths=.2,ax=ax,cbar_kws={"label":"Mean unemployment (%)"})
ax.set(title="State-wise monthly unemployment heatmap",xlabel="Month",ylabel="State")
ax.set_xticklabels([pd.Timestamp(t.get_text()).strftime("%b %y") for t in ax.get_xticklabels()],rotation=45,ha="right")
show_plot("11_state_month_heatmap")
note("Darker cells represent higher state-month unemployment rates. Blank cells indicate absent observations, not zero unemployment.")


## 12. Rural versus urban unemployment

In [ ]:
areas=set(df.Area.dropna().unique())
if {"Rural","Urban"}.issubset(areas):
    area_period=df[df.Area.isin(["Rural","Urban"])].groupby(["Area","COVID_Period"],as_index=False).Unemployment_Rate.mean()
    fig,ax=plt.subplots();sns.barplot(data=area_period,x="Area",y="Unemployment_Rate",hue="COVID_Period",ax=ax)
    ax.set(title="Rural vs urban unemployment by period",ylabel="Mean unemployment rate (%)",xlabel="Area")
    show_plot("12_rural_urban")
    display(area_period)
    note("Compare the rural and urban bars within each period; their averages are based on available observations, not population weights.")
else:
    note("Rural/urban breakdown is not available in the selected dataset; chart omitted rather than fabricated.")


## 13. Month-to-month change and labour-force participation trend

In [ ]:
monthly["Change_pp"]=monthly.Unemployment_Rate.diff()
fig,ax=plt.subplots(); ax.bar(monthly.Year_Month,monthly.Change_pp,width=22,color=np.where(monthly.Change_pp>=0,"#bd6a52","#3a8d83"))
ax.axhline(0,color="black",lw=.7);ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"));ax.tick_params(axis="x",rotation=40)
ax.set(title="Month-to-month change in mean unemployment",xlabel="Month",ylabel="Change (percentage points)")
show_plot("13_monthly_change")
if monthly.Change_pp.notna().any():
    max_change=monthly.loc[monthly.Change_pp.idxmax()]
    note(f"The largest monthly increase is {max_change.Change_pp:+.2f} percentage points in {max_change.Year_Month:%B %Y}.")
if "Labour_Participation_Rate" in df:
    lfpr=df.groupby("Year_Month",as_index=False).Labour_Participation_Rate.mean()
    fig,ax=plt.subplots();ax.plot(lfpr.Year_Month,lfpr.Labour_Participation_Rate,marker="o",color="#6d5c9d")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"));ax.tick_params(axis="x",rotation=40)
    ax.set(title="Monthly mean labour-force participation rate",xlabel="Month",ylabel="Labour participation rate (%)")
    show_plot("14_labour_participation")
    note("A falling unemployment rate need not indicate labour-market improvement if labour-force participation also falls.")


## 14. Statistical report and data-grounded conclusions

In [ ]:
summary=pd.Series({
"Observations":len(df),"States":df.State.nunique(),"First date":str(df.Date.min().date()),
"Last date":str(df.Date.max().date()),"Mean unemployment (%)":round(df.Unemployment_Rate.mean(),2),
"Median unemployment (%)":round(df.Unemployment_Rate.median(),2),
"Maximum unemployment (%)":round(df.Unemployment_Rate.max(),2),
"Minimum unemployment (%)":round(df.Unemployment_Rate.min(),2),
"Pre-COVID mean (%)":round(period_means["Pre-COVID"],2),
"COVID-affected mean (%)":round(period_means["COVID-affected"],2),
"COVID-period change (pp)":round(change,2),
"Highest-average-unemployment state":state_avg.index[0],
"Largest-increase state":changes.index[0] if len(changes) else "Not available",
"Highest-mean-unemployment month":peak.Year_Month.strftime("%B %Y")})
display(summary.to_frame("Value"))
summary.to_csv("India_Unemployment_Summary.csv",header=True)
conclusion=f"""## Key findings and conclusion

- The dataset contains **{len(df):,} observations** from **{df.State.nunique()} states/regions**, spanning **{df.Date.min():%B %Y}–{df.Date.max():%B %Y}**.
- The highest mean unemployment month was **{peak.Year_Month:%B %Y}**, at **{peak.Unemployment_Rate:.2f}%** across available reporting observations.
- The pre-COVID mean was **{period_means['Pre-COVID']:.2f}%** and the COVID-affected mean was **{period_means['COVID-affected']:.2f}%**, a **{change:+.2f} percentage-point** difference.
- **{state_avg.index[0]}** had the highest average observed unemployment rate (**{state_avg.iloc[0]:.2f}%**) over the available months.
- The three-state comparison and correlation heatmap are descriptive, and do not identify a causal effect of COVID-19.

**Limitations:** Coverage may differ by state and month; averages are unweighted, period lengths differ, missing-value imputation may influence results, and post-2020 recovery cannot be inferred without longer data. A robust causal study would require suitable controls and additional data.
"""
display(Markdown(conclusion))
Path("India_Unemployment_Conclusions.md").write_text(conclusion,encoding="utf-8")
print("Saved cleaned CSV, summary CSV, Markdown conclusions, and PNG charts in visualizations/.")


## 15. Export HTML report
Run the notebook with genuine data, then execute the following from a terminal: `jupyter nbconvert --to html --execute India_Unemployment_COVID19_EDA.ipynb --output India_Unemployment_Analysis_Report.html`. If already executed in Jupyter, omit `--execute`.